# Module 01: The BIDS Standard

The **Brain Imaging Data Structure (BIDS)** is a community standard for organising neuroimaging
datasets in a consistent, self-describing way. It was introduced in 2016 and is now supported
by hundreds of analysis tools.

## Why BIDS?

Before BIDS, every lab organised data differently. This made sharing and automated processing
extremely difficult. BIDS solves this by specifying:

- A **directory hierarchy** that separates subjects, sessions, and modalities.
- A **file-naming convention** using key-value pairs (e.g. `sub-01_task-rest_bold.nii.gz`).
- **JSON sidecar files** that store acquisition parameters alongside each image.
- **TSV files** for tabular data (events, participants, sessions).

Tools that understand BIDS include fMRIPrep, MRIQC, nilearn, pybids, and many others.

## Learning Objectives

1. Describe the BIDS directory structure and file-naming conventions.
2. Understand JSON sidecar files and events TSV files.
3. Use pybids to query a BIDS dataset programmatically.
4. Read and interpret `participants.tsv` and events files.

In [ ]:
import os
import json
import pathlib
import pandas as pd

# Path to the example BIDS dataset bundled with this repository
REPO_ROOT = pathlib.Path("__file__").resolve().parent.parent
BIDS_ROOT = REPO_ROOT / "data" / "example_bids"

print("Repository root  :", REPO_ROOT)
print("BIDS dataset path:", BIDS_ROOT)
print("BIDS path exists :", BIDS_ROOT.exists())

## 1. BIDS Directory Structure

A minimal BIDS dataset looks like this:

```
my_dataset/
├── dataset_description.json   ← required: dataset name, BIDS version
├── participants.tsv            ← subject-level metadata (age, sex, …)
├── participants.json           ← data dictionary for participants.tsv
├── README
└── sub-01/
    └── ses-01/                 ← optional session level
        ├── anat/
        │   └── sub-01_ses-01_T1w.nii.gz
        └── func/
            ├── sub-01_ses-01_task-rest_bold.nii.gz
            ├── sub-01_ses-01_task-rest_bold.json
            └── sub-01_ses-01_task-rest_events.tsv
```

Key entities in BIDS file names:

| Entity | Example | Meaning |
|--------|---------|----------|
| `sub` | `sub-01` | Subject identifier |
| `ses` | `ses-01` | Session identifier |
| `task` | `task-rest` | Task name |
| `run` | `run-1` | Run number |
| `acq` | `acq-multiband` | Acquisition variant |
| `space` | `space-MNI152` | Coordinate space |

In [ ]:
def print_tree(root: pathlib.Path, prefix: str = "", max_depth: int = 4, depth: int = 0) -> None:
    """Recursively print a directory tree up to max_depth."""
    if depth > max_depth:
        return
    entries = sorted(root.iterdir()) if root.is_dir() else []
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, prefix + extension, max_depth, depth + 1)

if BIDS_ROOT.exists():
    print(str(BIDS_ROOT))
    print_tree(BIDS_ROOT)
else:
    print("⚠️   Example BIDS dataset not found at", BIDS_ROOT)
    print("    The demo cells below will use a programmatically created structure instead.")

## 2. The dataset_description.json File

Every BIDS dataset must contain a `dataset_description.json` at the root level. This file
documents the dataset name, the BIDS version used, and optional provenance information.

In [ ]:
desc_path = BIDS_ROOT / "dataset_description.json"

if desc_path.exists():
    with open(desc_path) as f:
        description = json.load(f)
    print(json.dumps(description, indent=2))
else:
    # Show an example structure
    example = {
        "Name": "Example fMRI Dataset",
        "BIDSVersion": "1.9.0",
        "License": "CC0",
        "Authors": ["Researcher One", "Researcher Two"],
        "DatasetType": "raw",
    }
    print("Example dataset_description.json:")
    print(json.dumps(example, indent=2))

## 3. JSON Sidecar Files

Each NIfTI image in BIDS is accompanied by a JSON **sidecar file** with the same name but a
`.json` extension. The sidecar stores MRI acquisition parameters that cannot be stored inside
the NIfTI header itself, such as `RepetitionTime`, `EchoTime`, `PhaseEncodingDirection`, and
`TaskName`.

Sidecars are hierarchical: a file at the dataset root applies to all matching files unless
overridden at the subject or session level.

In [ ]:
# Search for bold JSON sidecars in the BIDS dataset
sidecar_paths = list(BIDS_ROOT.glob("**/*_bold.json")) if BIDS_ROOT.exists() else []

if sidecar_paths:
    path = sidecar_paths[0]
    print("Reading:", path.relative_to(BIDS_ROOT))
    with open(path) as f:
        sidecar = json.load(f)
    print(json.dumps(sidecar, indent=2))
else:
    # Show a representative example
    example_sidecar = {
        "TaskName": "rest",
        "RepetitionTime": 2.0,
        "EchoTime": 0.03,
        "FlipAngle": 90,
        "PhaseEncodingDirection": "j-",
        "SliceTiming": [0.0, 0.0625, 0.125, "..."],
        "MultibandAccelerationFactor": 4,
    }
    print("Example *_bold.json sidecar:")
    print(json.dumps(example_sidecar, indent=2))

## 4. Events Files

For task-based fMRI, BIDS requires an `*_events.tsv` file alongside each functional image. This
tab-separated file must contain at minimum two columns:

- `onset` — start time of the event in seconds from the beginning of the run.
- `duration` — duration of the event in seconds.

Additional columns (e.g. `trial_type`, `response_time`) are allowed and encouraged.

In [ ]:
events_paths = list(BIDS_ROOT.glob("**/*_events.tsv")) if BIDS_ROOT.exists() else []

if events_paths:
    events_df = pd.read_csv(events_paths[0], sep="\t")
    print("Events file:", events_paths[0].relative_to(BIDS_ROOT))
else:
    # Construct a synthetic events file for demonstration
    import numpy as np
    rng = np.random.default_rng(42)
    n_trials = 20
    onsets = np.cumsum(rng.uniform(8, 14, n_trials))
    events_df = pd.DataFrame({
        "onset": np.round(onsets, 2),
        "duration": rng.choice([1.0, 2.0], n_trials),
        "trial_type": rng.choice(["face", "house", "object"], n_trials),
        "response_time": np.round(rng.uniform(0.3, 1.5, n_trials), 3),
    })
    print("Synthetic events file:")

print(events_df.head(10).to_string(index=False))
print(f"\nTotal trials: {len(events_df)}")
if "trial_type" in events_df.columns:
    print("Condition counts:")
    print(events_df["trial_type"].value_counts().to_string())

## 5. participants.tsv

The `participants.tsv` file at the dataset root contains one row per subject with demographic
and experimental variables. A companion `participants.json` file documents each column.

In [ ]:
participants_path = BIDS_ROOT / "participants.tsv"

if participants_path.exists():
    participants_df = pd.read_csv(participants_path, sep="\t")
else:
    import numpy as np
    rng = np.random.default_rng(0)
    n_subjects = 10
    participants_df = pd.DataFrame({
        "participant_id": [f"sub-{i:02d}" for i in range(1, n_subjects + 1)],
        "age": rng.integers(20, 40, n_subjects),
        "sex": rng.choice(["M", "F"], n_subjects),
        "handedness": rng.choice(["R", "L", "A"], n_subjects, p=[0.85, 0.10, 0.05]),
        "group": rng.choice(["control", "patient"], n_subjects),
    })
    print("Synthetic participants.tsv:")

print(participants_df.to_string(index=False))
print(f"\nDataset: {len(participants_df)} subjects")
if "sex" in participants_df.columns:
    print("Sex distribution:")
    print(participants_df["sex"].value_counts().to_string())

## 6. Querying a BIDS Dataset with pybids

[pybids](https://bids-standard.github.io/pybids/) provides a Python API for querying BIDS
datasets. Once you have loaded a `BIDSLayout`, you can search for files using any combination
of entities (subject, session, task, suffix, extension, …).

In [ ]:
try:
    from bids import BIDSLayout

    if BIDS_ROOT.exists():
        layout = BIDSLayout(str(BIDS_ROOT))

        print("=== Dataset summary ===")
        print(layout)

        print("\n=== Subjects ===")
        print(layout.get_subjects())

        print("\n=== Tasks ===")
        print(layout.get_tasks())

        print("\n=== All bold files ===")
        bold_files = layout.get(suffix="bold", extension=".nii.gz")
        for f in bold_files[:5]:
            print(" ", f.relpath)

        print("\n=== Metadata for first bold file ===")
        if bold_files:
            meta = layout.get_metadata(bold_files[0].path)
            for key, val in list(meta.items())[:8]:
                print(f"  {key}: {val}")
    else:
        print("⚠️   Example BIDS dataset not found. Point BIDS_ROOT to a real BIDS dataset.")

except ImportError:
    print("⚠️   pybids is not installed.")
    print("    Install it with:  conda install -c conda-forge pybids")

## 7. Filtering Files with pybids

pybids makes it easy to select files for a specific subject, task, or run — the building block
for any automated analysis pipeline.

In [ ]:
try:
    from bids import BIDSLayout

    if BIDS_ROOT.exists():
        layout = BIDSLayout(str(BIDS_ROOT))
        subjects = layout.get_subjects()

        if subjects:
            first_sub = subjects[0]
            print(f"Files for subject {first_sub}:")
            files = layout.get(subject=first_sub, extension=[".nii.gz", ".json", ".tsv"])
            for f in files:
                print(" ", f.relpath)

            # Query for events files for this subject
            print(f"\nEvents files for subject {first_sub}:")
            events_files = layout.get(subject=first_sub, suffix="events")
            for ef in events_files:
                print(" ", ef.relpath)
                df = pd.read_csv(ef.path, sep="\t")
                print(df.head(3).to_string(index=False))
    else:
        print("⚠️   Skipping: example BIDS dataset not found.")

except ImportError:
    print("⚠️   pybids not available — skipping query demo.")

## Summary

In this notebook you have:

- Learned the BIDS directory hierarchy and file-naming conventions.
- Explored `dataset_description.json`, JSON sidecar files, events TSV files, and `participants.tsv`.
- Used pybids to load a BIDS dataset and query files by entity.

### Key Takeaways

- BIDS makes datasets **self-describing** — every file contains the metadata needed to process it.
- BIDS-compatible tools (fMRIPrep, MRIQC, nilearn) can ingest any BIDS dataset without custom
  configuration.
- pybids is the Python interface for programmatic access to BIDS metadata and file paths.

### Next Steps

Proceed to **Module 02: fMRI Preprocessing** to learn how raw BIDS data is processed with
fMRIPrep.